# Phase 0 — Reproduce NLBSE'27 CodeBERT baseline

Runs on **Colab T4**. All results are saved to **Google Drive** so nothing is lost when the session ends.

Drive folder used:
`MyDrive/NLBSE27_DistilVulSATD/`

Expected: avg F1 ≈ 0.910, score ≈ 0.7299

In [3]:
# Cell 1 — install + connect Google Drive (persistent storage)
!pip -q install "transformers>=4.36.0" "datasets>=2.16.0" "pandas" "numpy" "tqdm" "accelerate" "safetensors"

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/NLBSE27_DistilVulSATD')
(DRIVE_ROOT / 'results').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'logs').mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT =', DRIVE_ROOT)
print('Everything important will be saved here (survives Colab session end).')

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/NLBSE27_DistilVulSATD
Everything important will be saved here (survives Colab session end).


In [4]:
# Cell 2 — imports + GPU check
import time
import json
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from transformers import RobertaForSequenceClassification, RobertaTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
assert device.type == 'cuda', 'Enable GPU: Runtime → Change runtime type → T4 GPU'

device = cuda


In [5]:
# Cell 3 — load public test set from Hugging Face
QUICK = False  # True = smoke test on 512 rows; False = full official eval
MODEL = 'NLBSE/nlbse27-code-comment-classification-baseline'
MAX_AVG_RUNTIME = 1000
MAX_AVG_FLOPS = 3_000_000

ds = load_dataset('NLBSE/nlbse27-code-comment-classification')
test = ds['test'].to_pandas()
if QUICK:
    test = test.head(512).reset_index(drop=True)

print('test size =', len(test))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/530 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.15MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/214586 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30655 [00:00<?, ? examples/s]

test size = 30655


In [6]:
# Cell 4 — run baseline + SAVE results to Drive
tokenizer = RobertaTokenizer.from_pretrained(MODEL)
model = RobertaForSequenceClassification.from_pretrained(MODEL)
model.to(device)
model.eval()

repetitions = 1 if QUICK else 10
batch_size = 32 if QUICK else 600
measure_flops = (not QUICK)

tokenized = tokenizer(
    test['Function'].tolist(),
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=512,
)
loader = DataLoader(
    TensorDataset(tokenized['input_ids'], tokenized['attention_mask']),
    batch_size=batch_size,
)

predictions = []
total_time = 0.0
total_flops = 0.0

pbar = tqdm(total=len(test), desc='Predicting')
for batch in loader:
    input_ids, attention_mask = [x.to(device, non_blocking=True) for x in batch]
    begin = time.time()
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            if measure_flops:
                with torch.profiler.profile(with_flops=True) as prof:
                    for _ in range(repetitions):
                        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                total_flops += sum(k.flops for k in prof.key_averages()) / 1e9
            else:
                for _ in range(repetitions):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    total_time += time.time() - begin

    probs = torch.sigmoid(outputs.logits)
    batch_pred = (probs > 0.5).int().cpu().numpy().tolist()
    predictions.extend(batch_pred)
    pbar.update(len(batch_pred))
    del input_ids, attention_mask, outputs, probs
    torch.cuda.empty_cache()
pbar.close()

labels = np.array(test['Label'].tolist())
predictions = np.array(predictions)

scores = []
for i, name in enumerate(['Vulnerability', 'MAT']):
    tp = int(np.sum((predictions[:, i] == 1) & (labels[:, i] == 1)))
    fp = int(np.sum((predictions[:, i] == 1) & (labels[:, i] == 0)))
    fn = int(np.sum((predictions[:, i] == 0) & (labels[:, i] == 1)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    scores.append({'cat': name, 'precision': precision, 'recall': recall, 'f1': f1})

scores_df = pd.DataFrame(scores)
avg_f1 = float(scores_df.f1.mean())
avg_runtime = total_time / repetitions
avg_flops = total_flops / repetitions if measure_flops else 0.0
submission = (
    0.6 * avg_f1
    + 0.2 * max((MAX_AVG_RUNTIME - avg_runtime) / MAX_AVG_RUNTIME, 0)
    + 0.2 * max((MAX_AVG_FLOPS - avg_flops) / MAX_AVG_FLOPS, 0)
)

payload = {
    'model': MODEL,
    'quick': QUICK,
    'n_examples': len(test),
    'avg_f1': avg_f1,
    'avg_runtime': avg_runtime,
    'avg_flops': avg_flops,
    'submission_score': float(submission),
    'per_class': scores,
}

out_json = DRIVE_ROOT / 'results' / 'baseline_eval.json'
out_csv = DRIVE_ROOT / 'results' / 'baseline_scores.csv'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)
scores_df.to_csv(out_csv, index=False)

print('----------------RESULTS--------------------')
print(scores_df.to_string(index=False))
print('avg F1:', avg_f1)
print('avg runtime:', avg_runtime)
print('avg GFLOPs:', avg_flops)
print('submission_score:', round(submission, 4))
print('SAVED TO DRIVE:')
print(' ', out_json)
print(' ', out_csv)

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting: 100%|██████████| 30655/30655 [43:18<00:00, 11.80it/s]

----------------RESULTS--------------------
          cat  precision   recall       f1
Vulnerability   0.942651 0.935336 0.938979
          MAT   0.978261 0.801980 0.881393
avg F1: 0.9101860725779012
avg runtime: 259.2053787231445
avg GFLOPs: 2666509.7131617307
submission_score: 0.7165
SAVED TO DRIVE:
  /content/drive/MyDrive/NLBSE27_DistilVulSATD/results/baseline_eval.json
  /content/drive/MyDrive/NLBSE27_DistilVulSATD/results/baseline_scores.csv
